# Train contextual prediction model
Loads a validated processed CSV, builds contexts, fits experts, and evaluates a holdout set.


In [ ]:
from pathlib import Path
import pandas as pd, yaml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from eta_digital.contexts import ContextModel
from eta_digital.experts import ContextualMixtureOfExperts
from eta_digital.data import validate_training_frame


In [ ]:
PROJECT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = PROJECT.parent / 'data/offline/processed/training.csv'
with open(PROJECT/'configs/contexts.yaml') as f: contexts_cfg=yaml.safe_load(f)
with open(PROJECT/'configs/model.yaml') as f: model_cfg=yaml.safe_load(f)
frame=validate_training_frame(pd.read_csv(DATA))
train,test=train_test_split(frame,test_size=.2,random_state=42)
predictor=ContextualMixtureOfExperts(ContextModel.from_config(contexts_cfg),model_cfg['features'],model_cfg['outputs'],model_cfg['ridge_alpha'],model_cfg['minimum_context_weight']).fit(train)
pred=predictor.predict(test)
metrics={f'rmse_{c}':mean_squared_error(test[c],pred[c])**.5 for c in model_cfg['outputs']}
metrics
